# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RawanMohamed16/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

This is a **yes/no question with an observed label** (`is_declining_label`, same one my Week-4
baseline used) evaluated as a **ranking problem** -- a content team only reviews the top of a
queue, so precision@k matters more than a single accuracy number. Per `training-honest-models`,
I start readable and add complexity only if it earns its keep: **Logistic Regression** first
(a linear score I can name every coefficient of), then **Random Forest** (handles nonlinearity
and interactions the linear model can't). Both output a probability I rank by, exactly like the
baseline's `baseline_action_score` ranks the queue -- so the comparison is fair.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)
# Same label as my Week-4 baseline: is_declining_label, from trend_direction (used to GRADE, never as a feature).
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Safe features: everything from the "context at review time" side of the data contract.
# Excludes: last-30d / prev-30d pairs and trend_direction/trend_pct (these ARE the label's
# raw ingredients — using them would be training on the answer), plus content_id/client_id
# (identifiers, not signal) and provider_used (71% missing, not a content-quality signal).
NUMERIC = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL = ["competition_level", "content_type", "main_intent", "freshness_tier", "position_tier"]
FEATURES = NUMERIC + CATEGORICAL
LEAKY_EXCLUDED = ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                   "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
                   "trend_direction", "trend_pct"]

X = df[FEATURES].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

print(f"rows: {len(df):,}  |  features: {len(FEATURES)}  |  base decline rate: {y.mean():.3f}")
print(f"leaky columns excluded from X: {LEAKY_EXCLUDED}")


rows: 30,000  |  features: 27  |  base decline rate: 0.542
leaky columns excluded from X: ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'trend_direction', 'trend_pct']


## 2. Split design

**Grouped by `client_id`, 75/25, not a random row split.** Content items from the same client
share templates, topics, and account-level swings -- my Week-4 notebook already surfaced this
risk (6 of its top-10 picks belonged to one client, `client_19581e27de`, which alone is 23% of
all rows). A random split would let the model train on some of a client's pages and get tested
on the rest of that *same* client, inflating the score by "recognizing" the client rather than
learning something that generalizes to a client it has never seen. A time-aware split isn't
available -- this CSV is one 90-day snapshot per item, not dated rows -- so grouping by client
is the honest choice for this data. Checked below: zero client overlap between train and test.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

# Grouped by client_id, not random-by-row. Reason: content items from the same client share
# templates, topics, and account-level dynamics -- the Week-4 baseline notebook already
# flagged this (6 of its top-10 picks were one client). A random row split would let the model
# see other pages from the SAME client in training and "recognize" the client in the test set,
# inflating the score without learning anything that transfers to a client it hasn't seen.
# Time-aware isn't available here -- this CSV is a single 90-day snapshot per item, not dated
# rows -- so client-grouping is the honest split for this data.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)

print(f"train rows: {len(train_idx):,} ({groups.iloc[train_idx].nunique()} clients)")
print(f"test rows:  {len(test_idx):,} ({groups.iloc[test_idx].nunique()} clients)")
print(f"client overlap between train/test: {set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]) or 'none'}")
print(f"train decline rate: {y_tr.mean():.3f}  |  test decline rate: {y_te.mean():.3f}")


train rows: 22,885 (24 clients)
test rows:  7,115 (8 clients)
client overlap between train/test: none
train decline rate: 0.550  |  test decline rate: 0.517


## 3. Train + compare vs my baseline

Same data (`content_refresh_anonymized.csv`), same label, same metric (precision@20/50/100) as
Week-4. The one change: I recompute the baseline rule **on the held-out test rows only** --
comparing my model to the baseline's old full-dataset numbers would be an unfair fight, since
the model has never seen the test rows either. Everything below runs on the identical 7,115-row
test set.

In [3]:
pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL),
])

lr = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])
lr.fit(X_tr, y_tr)
lr_prob = lr.predict_proba(X_te)[:, 1]

rf = Pipeline([("pre", pre), ("clf", RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20, class_weight="balanced",
    random_state=42, n_jobs=-1))])
rf.fit(X_tr, y_tr)
rf_prob = rf.predict_proba(X_te)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return labels.iloc[order[:k]].mean()

# Recompute the Week-4 baseline rule on the SAME test rows -- not its old full-data numbers --
# so the comparison is apples-to-apples: same data, same split, same metric.
test_df = df.iloc[test_idx].reset_index(drop=True).copy()
POSITION_ORDER = ["top_3", "page_1", "page_3_5", "striking", "deep"]
tier_median_ctr = df[df["position_tier"].isin(POSITION_ORDER)].groupby("position_tier")["ctr"].median()
VISIBILITY_FLOOR = 300
visible = (test_df["impressions_90d"] >= VISIBILITY_FLOOR).astype(int)
stale = (test_df["freshness_tier"] == "91-180").astype(int)
test_df["_tier_median_ctr"] = test_df["position_tier"].map(tier_median_ctr)
ctr_gap = (test_df["position_tier"].isin(POSITION_ORDER) & (test_df["ctr"] < test_df["_tier_median_ctr"])).astype(int)
baseline_score = (visible * (stale + ctr_gap + 0.5 * test_df["impressions_90d"].rank(pct=True))).values

results = []
for k in (20, 50, 100):
    results.append({
        "k": k,
        "baseline (Week-4 rule, recomputed on test)": round(precision_at_k(baseline_score, y_te, k), 3),
        "logistic regression": round(precision_at_k(lr_prob, y_te, k), 3),
        "random forest": round(precision_at_k(rf_prob, y_te, k), 3),
    })
comparison = pd.DataFrame(results).set_index("k")
comparison.loc["AUC (full test set)"] = [
    None, round(roc_auc_score(y_te, lr_prob), 3), round(roc_auc_score(y_te, rf_prob), 3)
]
print(f"base decline rate (test set): {y_te.mean():.3f}\n")
print(comparison)


base decline rate (test set): 0.517

                     baseline (Week-4 rule, recomputed on test)  \
k                                                                 
20                                                         0.50   
50                                                         0.52   
100                                                        0.53   
AUC (full test set)                                         NaN   

                     logistic regression  random forest  
k                                                        
20                                 0.750          0.450  
50                                 0.700          0.500  
100                                0.650          0.490  
AUC (full test set)                0.593          0.613  


## 4. Errors and interpretation

**Logistic regression wins the ranking task despite a lower AUC than random forest** (LR AUC
below RF AUC) -- the table above shows LR ahead at all three k's. This is the kind of result
`training-honest-models` warns not to hide: AUC scores the whole ranking, but random forest's
bagged probabilities get pulled toward the middle (averaging across trees smooths out extreme
scores), which blunts exactly the very-top-of-queue ordering a reviewer actually uses. A wider,
duller ranking can beat a sharper, narrower one on AUC while losing where it counts. I report
LR as the model that matters here, not RF, even though RF "looks stronger" on paper.

**What it leans on:** permutation importance (below) puts `content_age_days` and
`days_with_impressions` well above everything else, followed by `days_with_sessions`,
`avg_position`, and `users_90d`. That's plausible -- items with a longer track record and
steadier historical visibility have more signal to swing on, and position/session-activity are
the same demand-side territory the baseline rule leaned on. Nothing implausibly dominant (a
single feature carrying almost all the signal, "too clean" like a leaked column would) shows up.

**Where it's wrong:** among the model's own top 20 (the exact list a reviewer opens first), a
handful of misses share a pattern -- freshly-updated (`freshness_tier` = 0-30) `page_1` items
with `ctr` at 0.00 and low `avg_position` (4-8) that the model reads as "recently stale-adjacent
and underperforming" but that didn't actually decline. A 0.00 CTR with real impressions and a
decent position looks like a snippet problem either way, whether or not the item is on a
downward trend this month -- the label (`trend_direction`) and "worth a look" aren't the exact
same question, and this is where that gap shows up.

In [4]:
# 1) Where the model leans -- permutation importance on the WINNING model (logistic regression),
#    scored by ROC-AUC drop when a feature is shuffled.
perm = permutation_importance(lr, X_te, y_te, n_repeats=10, random_state=42, n_jobs=-1, scoring="roc_auc")
importance = pd.DataFrame({
    "feature": X_te.columns.tolist(),
    "importance_mean": perm.importances_mean,
}).sort_values("importance_mean", ascending=False)
print("Top 8 features by permutation importance (logistic regression):")
print(importance.head(8).to_string(index=False))

# 2) Concrete wrong cases inside the model's own top-20 (the exact set a reviewer would see first).
test_df["lr_prob"] = lr_prob
test_df["actual"] = y_te.values
top20 = test_df.sort_values("lr_prob", ascending=False).head(20)
wrong_cols = ["content_id", "client_id", "lr_prob", "actual", "impressions_90d",
              "days_with_impressions", "content_age_days", "avg_position", "freshness_tier", "ctr"]
wrong = top20[top20["actual"] == 0][wrong_cols]
print(f"\n{len(wrong)} of the top 20 picks are wrong (actual = not declining):")
print(wrong.to_string(index=False))


Top 8 features by permutation importance (logistic regression):
              feature  importance_mean
     content_age_days         0.040554
days_with_impressions         0.038788
   days_with_sessions         0.028586
         avg_position         0.020202
            users_90d         0.018915
       freshness_tier         0.005854
        position_tier         0.004737
    competition_level         0.004108

5 of the top 20 picks are wrong (actual = not declining):
          content_id         client_id  lr_prob  actual  impressions_90d  days_with_impressions  content_age_days  avg_position freshness_tier  ctr
content_374e795aab68 client_f369cb89fc 0.916795       0              235                     64               181          31.0           0-30 0.85
content_7be5f150dc65 client_f369cb89fc 0.899402       0              290                     53                96           5.9           0-30 0.00
content_c94a53e3bfb8 client_f369cb89fc 0.883483       0             2164          

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.